# Лабораторна робота 5 — Логістична регресія для аналізу тональності

**Набори даних:** `amazon_baby_subset.csv`, `important_words.json`  
**Обмеження:** scikit-learn-класифікатори **не дозволені** для базових завдань.

## Налаштування

In [1]:
import sys
!{sys.executable} -m pip install numpy pandas matplotlib --quiet


In [2]:
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt

%matplotlib inline


## Теоретичне підґрунтя

Сигмоїдна функція:
```
P(y = +1 | x, w) = 1 / (1 + exp(−wᵀ h(x)))
```
Похідна логарифму правдоподібності відносно wⱼ:
```
d(ll)/dwⱼ = dot( hⱼ,  1[y=+1] − P(y=+1|x,w) )
```
де 1[y=+1] = 1, якщо мітка +1, інакше 0.

---
## Завдання 1 — Підготовка ознак

1. Завантажте `amazon_baby_subset.csv`. Видаліть рядки з відсутніми відгуками. Вилучіть відгуки з `rating == 3`. Створіть стовпець `sentiment`: **+1** якщо `rating >= 4`, інакше **−1**.
2. Завантажте `important_words.json` (193 слова). Для кожного слова додайте стовпець до DataFrame із підрахунком його входжень у очищений текст відгуку.
3. Повідомте, скільки відгуків залишилось та який баланс класів.

In [3]:
# Завантаження набору даних
products = pd.read_csv('amazon_baby_subset.csv')

# Видаліть рядки з відсутніми відгуками та нейтральними рейтингами
products = products.dropna(subset=['review', 'rating']).copy()
products = products[products['rating'] != 3].copy()

# Створіть стовпець sentiment: +1 якщо рейтинг >= 4, інакше -1
products['sentiment'] = np.where(products['rating'] >= 4, +1, -1)
products['sentiment'] = products['sentiment'].astype(int)

print(f'Усього відгуків : {len(products)}')
print(f'Позитивні (+1)  : {(products["sentiment"] == 1).sum()}')
print(f'Негативні (-1)  : {(products["sentiment"] == -1).sum()}')


Усього відгуків : 17310
Позитивні (+1)  : 17310
Негативні (-1)  : 0


In [4]:
# Завантаження важливих слів
with open('important_words.json') as f:
    important_words = json.load(f)
print(f'Розмір словника: {len(important_words)} слів')


Розмір словника: 193 слів


In [5]:
# Очищення тексту відгуків (видалення пунктуації, приведення до нижнього регістру)
import string
from collections import Counter

products['review_clean'] = (
    products['review']
    .fillna('')
    .str.replace(f'[{string.punctuation}]', '', regex=True)
    .str.lower()
)

important_word_set = set(important_words)
word_count_rows = []

# Додайте стовпець підрахунку слів для кожного важливого слова
for text in products['review_clean']:
    counts = Counter(token for token in text.split() if token in important_word_set)
    word_count_rows.append({word: counts.get(word, 0) for word in important_words})

word_count_df = pd.DataFrame(word_count_rows, index=products.index)
products = pd.concat([products, word_count_df], axis=1)

print('Приклад підрахунку слів:')
products[important_words[:5]].head(3)


Приклад підрахунку слів:


,baby,one,great,love,use
0,0,0,1,0,0
1,0,0,0,0,0
2,1,0,0,0,0


---
## Завдання 2 — Побудова матриці ознак

Реалізуйте `get_feature_matrix(df, word_list)`, яка:
1. Створює масив NumPy зі стовпцем одиниць (вільний член), за яким іде по одному стовпцю для кожного слова зі списку.
2. Повертає `(feature_matrix, sentiment_array)`, де `sentiment_array` містить +1 або −1.

Перевірка: `feature_matrix` має форму `(N, 194)`.

In [6]:
def get_feature_matrix(df, word_list):
    """
    Будує матрицю ознак та вектор міток тональності.

    Повертає
    -------
    feature_matrix  : np.ndarray, shape (n, len(word_list)+1)
    sentiment_array : np.ndarray, shape (n,)  значення {+1, -1}
    """
    # Intercept column of 1s
    intercept = np.ones((len(df), 1), dtype=float)

    # Word feature columns
    word_features = df[word_list].to_numpy(dtype=float)

    feature_matrix = np.hstack((intercept, word_features))

    sentiment_array = df['sentiment'].to_numpy(dtype=int)
    return feature_matrix, sentiment_array


In [7]:
feature_matrix, sentiment = get_feature_matrix(products, important_words)
print(f'Розмір feature_matrix: {feature_matrix.shape}')  # очікується (N, 194)


Розмір feature_matrix: (17310, 194)


---
## Завдання 3 — Сигмоїдна функція та передбачення

Реалізуйте `predict_probability(feature_matrix, coefficients)`, що обчислює сигмоїдну функцію для кожного рядка. Результат — масив NumPy зі значеннями у (0, 1).

In [8]:
def predict_probability(feature_matrix, coefficients):
    """
    Обчислює P(y = +1 | x, w) для кожного рядка.

    Повертає
    -------
    probabilities : np.ndarray, shape (n,), значення у (0, 1)
    """
    # Compute the linear score for each example
    score = np.dot(feature_matrix, coefficients)

    # Apply sigmoid
    score = np.clip(score, -500, 500)
    probabilities = 1.0 / (1.0 + np.exp(-score))

    return probabilities


### Перевірка — при нульових вхідних даних кожне передбачення має дорівнювати 0.5

In [9]:
zero_coeffs = np.zeros(feature_matrix.shape[1])
test_probs  = predict_probability(feature_matrix, zero_coeffs)
print(f'Усі передбачення 0.5: {np.allclose(test_probs, 0.5)}')


Усі передбачення 0.5: True


---
## Завдання 4 — Градієнтний підйом

Реалізуйте `logistic_regression(feature_matrix, sentiment, initial_coefficients, step_size, max_iter)`. На кожній ітерації:
1. Обчислюйте передбачення за допомогою `predict_probability()`.
2. Обчислюйте `errors = 1[y=+1] − predictions`.
3. Для кожного коефіцієнта j: `derivative = dot(feature_j, errors)`, потім `coeff[j] += step_size · derivative`.

Запустіть з: `initial_coefficients = np.zeros(194)`, `step_size = 1e-7`, `max_iter = 301`. Виводьте логарифм правдоподібності кожні 50 ітерацій — він має зростати монотонно.

In [10]:
def compute_log_likelihood(feature_matrix, sentiment, coefficients):
    """Допоміжна функція (надана) — обчислює логарифм правдоподібності для моніторингу."""
    indicator = (sentiment == +1).astype(float)
    scores    = np.dot(feature_matrix, coefficients)
    ll        = np.sum(
        indicator * scores - np.log(1.0 + np.exp(scores))
    )
    return ll


In [11]:
def logistic_regression(feature_matrix, sentiment,
                        initial_coefficients, step_size, max_iter):
    """
    Навчає ваги логістичної регресії методом градієнтного підйому.

    Повертає
    -------
    coefficients : np.ndarray, shape (n_features,)
    """
    coefficients = np.array(initial_coefficients, dtype=float)
    indicator = (sentiment == +1).astype(float)   # 1 якщо позитивний клас, інакше 0

    log_likelihood_values = []

    for itr in range(max_iter):
        # 1. Compute predictions P(y=+1 | x, w)
        predictions = predict_probability(feature_matrix, coefficients)

        # 2. Compute errors = indicator(y=+1) - predictions
        errors = indicator - predictions

        # 3. Update every coefficient
        derivatives = np.dot(feature_matrix.T, errors)
        coefficients += step_size * derivatives

         # Print log-likelihood every 50 iterations
        if itr % 50 == 0:
            ll = compute_log_likelihood(feature_matrix, sentiment, coefficients)
            log_likelihood_values.append(ll)
            print(f'Ітерація {itr:4d}  |  логарифм правдоподібності: {ll:.4f}')

    return coefficients


### Запуск моделі

In [12]:
coefficients = logistic_regression(
    feature_matrix, sentiment,
    initial_coefficients=np.zeros(feature_matrix.shape[1]),
    step_size=1e-7,
    max_iter=301
)


Ітерація    0  |  логарифм правдоподібності: -11974.3415
Ітерація   50  |  логарифм правдоподібності: -10883.7174
Ітерація  100  |  логарифм правдоподібності: -9977.9045
Ітерація  150  |  логарифм правдоподібності: -9215.4647
Ітерація  200  |  логарифм правдоподібності: -8565.2163
Ітерація  250  |  логарифм правдоподібності: -8004.0182
Ітерація  300  |  логарифм правдоподібності: -7514.5802


### Точність класифікації та базовий рівень

In [13]:
# Передбачте мітки класів (+1 якщо score > 0, інакше -1)
scores          = np.dot(feature_matrix, coefficients)
predictions     = np.where(scores > 0, +1, -1)

# Точність моделі
model_accuracy  = np.mean(predictions == sentiment)
print(f'Точність моделі   : {model_accuracy:.4f}')

# Базовий рівень мажоритарного класу
majority_class  = +1 if (sentiment == +1).sum() >= (sentiment == -1).sum() else -1
baseline_acc    = np.mean(majority_class == sentiment)
print(f'Базовий рівень    : {baseline_acc:.4f}  (завжди передбачає {majority_class})')


Точність моделі   : 1.0000
Базовий рівень    : 1.0000  (завжди передбачає 1)


### Висновок до основної частини

Модель логістичної регресії була реалізована вручну без використання `scikit-learn`: побудовано матрицю ознак, реалізовано сигмоїдну функцію, log-likelihood та градієнтний підйом. Log-likelihood зростає під час навчання, що підтверджує коректний напрям оновлення коефіцієнтів.

Однак у наданому файлі після видалення пропущених значень усі відгуки мають рейтинги 4 або 5, тому всі мітки `sentiment` дорівнюють `+1`. Через це точність моделі дорівнює базовому рівню мажоритарного класу, а задача класифікації фактично є виродженою: модель не має негативних прикладів, на яких могла б навчитися відрізняти негативні відгуки від позитивних.


---
## ✨ Бонус — Інтерпретація моделі

Зіставте кожне слово з його навченим коефіцієнтом. Виведіть 10 слів з найбільшими коефіцієнтами та 10 слів з найменшими.

Для одного слова з кожного списку знайдіть відгук у наборі даних, що його містить, і вкажіть передбачувану ймовірність моделі.

In [14]:
# Бонус — аналіз коефіцієнтів
# Підказка: створіть DataFrame зі стовпцями ['word', 'coefficient']
coef_table = pd.DataFrame({
    'word': important_words,
    'coefficient': coefficients[1:]
})

positive_words = coef_table.sort_values('coefficient', ascending=False).head(10)
negative_words = coef_table.sort_values('coefficient', ascending=True).head(10)

print('10 слів з найбільшими коефіцієнтами:')
display(positive_words)

print('10 слів з найменшими коефіцієнтами:')
display(negative_words)


10 слів з найбільшими коефіцієнтами:


,word,coefficient
0,baby,0.080332
1,one,0.077518
2,great,0.075588
4,use,0.058326
5,would,0.056601
3,love,0.056481
7,easy,0.051769
8,little,0.051175
6,like,0.050968
10,old,0.046130


10 слів з найменшими коефіцієнтами:


,word,coefficient
189,won,0.000098
168,returned,0.000940
113,return,0.000962
112,waste,0.001154
171,broke,0.001319
105,disappointed,0.001671
182,unit,0.001878
166,company,0.002211
180,working,0.002712
133,cheap,0.002930


In [15]:
# Бонус — приклад відгуку з передбачуваною ймовірністю

def show_review_example(word, max_chars=700):
    """Знаходить перший відгук зі словом і показує ймовірність позитивного класу."""
    matching_rows = products[products[word] > 0]

    if matching_rows.empty:
        print(f'Слово "{word}" не знайдено у відгуках.')
        return None

    idx = matching_rows.index[0]
    row_position = products.index.get_loc(idx)
    probability = predict_probability(feature_matrix[row_position:row_position + 1], coefficients)[0]
    predicted_class = +1 if probability >= 0.5 else -1
    actual_class = products.loc[idx, 'sentiment']
    review_text = products.loc[idx, 'review']

    print(f'Слово: {word}')
    print(f'Ймовірність позитивного класу: {probability:.4f}')
    print(f'Передбачений клас: {predicted_class}')
    print(f'Реальний клас: {actual_class}')
    print('Фрагмент відгуку:')
    print(review_text[:max_chars] + ('...' if len(review_text) > max_chars else ''))

    return probability

most_positive_word = positive_words.iloc[0]['word']
smallest_coefficient_word = negative_words.iloc[0]['word']

print('Приклад для слова з найбільшим коефіцієнтом')
positive_example_probability = show_review_example(most_positive_word)

print()
print('Приклад для слова з найменшим коефіцієнтом')
smallest_example_probability = show_review_example(smallest_coefficient_word)


Приклад для слова з найбільшим коефіцієнтом
Слово: baby
Ймовірність позитивного класу: 0.6592
Передбачений клас: 1
Реальний клас: 1
Фрагмент відгуку:
My daughter had her 1st baby over a year ago. She did receive and fill up a First Year Calendar. When her son was nearing his first birthday she was looking for a Second Year Calendar to record his milestones. Thanks to Amazon I was able to get this for her and she LOVES it. Tender sweet art work - helpful stickers - unique pages to fill. A nice keepsake. A wonderful gift for a one-year old!

Приклад для слова з найменшим коефіцієнтом
Слово: won
Ймовірність позитивного класу: 0.6644
Передбачений клас: 1
Реальний клас: 1
Фрагмент відгуку:
My baby is now almost 11 months old and is probably too big to really bathe in this tub anymore; however I continue to use it because it's just so easy to pop it on top of the kitchen sink and fill it up. He sits up in it now and splashes and enjoys bath-time. We bought several other bath tubs and this on

**Найбільш позитивні слова:** слова з найбільшими коефіцієнтами наведені у таблиці `positive_words`.  
**Слова з найменшими коефіцієнтами:** наведені у таблиці `negative_words`.  
**Спостереження:** у цьому файлі після очищення залишилися лише позитивні відгуки, тому модель не бачить прикладів негативного класу. Через це коефіцієнти не можна інтерпретувати як повноцінне розділення позитивної та негативної тональності: слова з найбільшими коефіцієнтами найсильніше підтримують позитивний клас, а слова з найменшими коефіцієнтами радше мають найслабший позитивний вплив, а не обов'язково негативний зміст.
